### Ingest races.csv file

In [0]:
%run ../00-common/01-environment-config

In [0]:
%run ../00-common/02-bronze_helper

In [0]:
source_file = f"{landing_forlder_path}/races.csv"
table_name = f"{catalog_name}.{bronze_schema}.races"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

races_schema = StructType(fields=[
    StructField("season", IntegerType(), True),
    StructField("round", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("raceName", StringType(), True),
    StructField("date", DateType(), True),
    StructField("circuitId", StringType(), True)
])

races_df = (spark.read.format("csv")
            .option("header", "true")
            .schema(races_schema)
            .option('mode', 'FAILFAST')
            .load(source_file)
            .select("*", "_metadata.file_path"))


In [0]:
display(races_df)

In [0]:
races_df_final = add_ingestion_metadata(races_df)

### Creating Delta Table

In [0]:
(
races_df_final.write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(table_name)
)

In [0]:
%sql 
SELECT * from formula1.bronze.races limit 10

In [0]:
display(spark.sql("SHOW TABLES IN formula1.bronze"))